In [0]:
use catalog classic_workspace;
use schema default;

In [0]:
DROP TABLE IF EXISTS customers;

CREATE OR REPLACE TABLE customers (
  customer_id  INT,
  name         STRING,
  email        STRING,
  tier         STRING,
  updated_at   TIMESTAMP
)
USING DELTA;

INSERT INTO customers VALUES
  (1, 'Alice Nguyen',  'alice@example.com',  'gold',     '2024-06-20 10:00:00'),
  (2, 'Bob Patel',     'bob@example.com',    'silver',   '2024-06-20 10:00:00'),
  (3, 'Carol Santos',  'carol@example.com',  'platinum', '2024-06-20 10:00:00'),
  (4, 'David Kim',     'david@example.com',  'bronze',   '2024-06-20 10:00:00'),
  (5, 'Eva Müller',    'eva@example.com',    'silver',   '2024-06-20 10:00:00'),
  (6, 'Frank Osei',    'frank@example.com',  'bronze',   '2024-06-20 10:00:00'),
  (7, 'Grace Lin',     'grace@example.com',  'gold',     '2024-06-20 10:00:00');

In [0]:
CREATE OR REPLACE TABLE erasure_requests (
  customer_id  INT,
  requested_at DATE
)
USING DELTA;

INSERT INTO erasure_requests VALUES
  (2, '2024-06-25'),
  (4, '2024-06-26'),
  (5, '2024-06-27');

In [0]:
DELETE FROM customers
WHERE customer_id IN (
  SELECT customer_id FROM erasure_requests
);

In [0]:
CREATE OR REPLACE TABLE customer_spend (
  customer_id  INT,
  total_spend  DOUBLE
)
USING DELTA;

INSERT INTO customer_spend VALUES
  (1, 8200.00),   -- Alice: high spender
  (3, 12500.00),  -- Carol: very high spender
  (6, 350.00);    -- Frank: low spender

In [0]:
UPDATE customers
SET    tier = 'platinum'
WHERE  customer_id IN (
  SELECT customer_id
  FROM   customer_spend
  WHERE  total_spend > 5000
);

In [0]:
CREATE OR REPLACE TABLE orders (
  order_id    INT,
  customer_id INT,
  amount      DOUBLE,
  order_date  DATE
)
USING DELTA;

In [0]:
%python
batch_1 = [
    (1001, 1, 199.99, "2024-07-01"),
    (1002, 3, 449.00, "2024-07-01"),
    (1003, 6, 89.50,  "2024-07-01"),
]

df = spark.createDataFrame(batch_1, ["order_id", "customer_id", "amount", "order_date"]).selectExpr(
    "CAST(order_id AS INT) AS order_id",
    "CAST(customer_id AS INT) AS customer_id",
    "amount",
    "CAST(order_date AS DATE) AS order_date"
)

df.write \
  .format("delta") \
  .mode("append") \
  .option("txnAppId", "orders_daily_pipeline") \
  .option("txnVersion", 1) \
  .saveAsTable("orders")

count = spark.table("orders").count()
print(f"Row count after write: {count}")

In [0]:
DESCRIBE HISTORY orders;